In [0]:
USER_SCHEMA = "dbr_dev.jvanderbrug"
FILE_PATH = "/Volumes/dbr_dev/jvanderbrug/raw_files/london_merged.csv"

df_raw = (spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(FILE_PATH)
)

clean_cols = [col_name.strip().replace(' ', '_').lower() for col_name in df_raw.columns]
df_bronze = df_raw.toDF(*clean_cols)

df_bronze.write.format("delta").mode("overwrite").saveAsTable(f"{USER_SCHEMA}.bikes_bronze")

In [0]:
from pyspark.sql.functions import col, sum as _sum, when

total_rows = df_bronze.count()
print(f"Total number of records: {total_rows}")

null_exprs = [_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in df_bronze.columns]
null_counts = df_bronze.agg(*null_exprs).collect()[0].asDict()

print("\nNull values:")
for c, null_cnt in null_counts.items():
    if null_cnt > 0:
        print(f" Column '{c}': {null_cnt} null values")

duplicates = total_rows - df_bronze.dropDuplicates().count()
print(f"\nDuplicated rows: {duplicates}")

invalid_cnt = df_bronze.filter(col("cnt") < 0).count()
extreme_temp = df_bronze.filter((col("t1") > 45) | (col("t1") < -20)).count()

print("\nBusiness rule validation:")
print(f" - Records with negative rental count: {invalid_cnt}")
print(f" - Extreme temperature anomalies (>45C or <-20C): {extreme_temp}")

In [0]:
import pyspark.sql.functions as F

df_silver_raw = spark.table(f"{USER_SCHEMA}.bikes_bronze")

df_silver_daily = (df_silver_raw
    .withColumn("date_only", F.to_date(F.col("timestamp")))
    .groupBy("date_only")
    .agg(
        F.sum("cnt").alias("total_daily_bikes"),
        F.round(F.max("t1"), 1).alias("max_temp_C"),
        F.round(F.min("t1"), 1).alias("min_temp_C"),
        F.round(F.avg("wind_speed"), 1).alias("avg_wind_speed")
    )
    .withColumn("temp_amplitude", F.round(F.col("max_temp_C") - F.col("min_temp_C"), 1))
)

display(df_silver_daily)

In [0]:
import requests
from pyspark.sql.types import StructType, StructField, StringType, BooleanType, DateType
from datetime import datetime

url = "https://www.gov.uk/bank-holidays.json"
response = requests.get(url)
events = response.json()['england-and-wales']['events']

processed_holidays = []
for event in events:
    event_date = datetime.strptime(event['date'], '%Y-%m-%d').date()
    name = event['title']
    
    has_bunting = event.get('bunting', False) 
    
    holiday_type = "Major Festivity (Decorated)" if has_bunting else "Standard Holiday"
    
    processed_holidays.append((event_date, name, has_bunting, holiday_type))

schema = StructType([
    StructField("holiday_date", DateType(), True),
    StructField("holiday_name", StringType(), True),
    StructField("is_decorated", BooleanType(), True),
    StructField("holiday_type", StringType(), True)
])

df_holidays = spark.createDataFrame(processed_holidays, schema=schema)

df_gold = df_silver_daily.join(
    df_holidays,
    df_silver_daily.date_only == df_holidays.holiday_date,
    how="left"
).drop("holiday_date") 

df_gold = df_gold.fillna({
    "holiday_name": "No Holiday",
    "is_decorated": False,
    "holiday_type": "Regular Day"
})

display(df_gold)

df_gold.write.format("delta").mode("overwrite").saveAsTable(f"{USER_SCHEMA}.bikes_gold")